# Cuaderno 8 - Transformaciones condicionales con `when()`, `then()` y `otherwise()`

## Antes de comenzar

En el cuaderno anterior utilizamos `with_columns()` para agregar y transformar columnas aplicando una misma regla a todas las filas.

Ahora vamos a trabajar con situaciones en las que el resultado depende de una condición. Por ejemplo:

```text
si precio > 10000
    "Alto"
en caso contrario
    "Normal"
```

Polars permite expresar este tipo de lógica mediante:

```text
when()
then()
otherwise()
```

Seguiremos trabajando con expresiones sobre columnas completas. No recorreremos las filas manualmente ni utilizaremos funciones Python fila por fila.


## 1. Preparar el entorno y los datos

Instalamos Polars, importamos la biblioteca y reconstruimos el DataFrame de ventas que venimos utilizando.


In [21]:
!pip install -q polars

In [22]:
import polars as pl

In [23]:
datos = {
    "producto": ["Cuaderno", "Lapicera", "Mochila", "Calculadora", "Regla", "Carpeta"],
    "categoria": ["Librería", "Librería", "Accesorios", "Tecnología", "Librería", "Librería"],
    "precio": [3500, 1200, 28500, 18500, 900, 4200],
    "cantidad": [4, 10, 2, 3, 6, 5],
}

df = pl.DataFrame(datos)
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


El DataFrame ejecutado tiene `shape: (6, 4)` y contiene las columnas `producto`, `categoria`, `precio` y `cantidad`. `precio` y `cantidad` son de tipo `Int64`. A partir de esta tabla construiremos nuevas variables cuyos valores dependerán de condiciones.


## 2. Observar primero la condición

Antes de clasificar los productos, observamos qué filas tienen un precio mayor que `10000`.


In [24]:
df.select(
    "producto",
    "precio",
    (pl.col("precio") > 10000).alias("supera_10000"),
)

producto,precio,supera_10000
str,i64,bool
"""Cuaderno""",3500,false
"""Lapicera""",1200,false
"""Mochila""",28500,true
"""Calculadora""",18500,true
"""Regla""",900,false
"""Carpeta""",4200,false


La salida tiene `shape: (6, 3)`. La columna `supera_10000` contiene `true` solamente para `Mochila` y `Calculadora`, cuyos precios son `28500` y `18500`.

Esta misma expresión booleana será utilizada ahora dentro de `when()`.


## 3. Una clasificación binaria

Queremos crear `nivel_precio` con dos valores posibles:

```text
precio > 10000  → "Alto"
resto           → "Normal"
```


In [25]:
df.with_columns(
    nivel_precio=(
        pl.when(
            pl.col("precio") > 10000
        )
        .then(pl.lit("Alto"))
        .otherwise(pl.lit("Normal"))
    )
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Normal"""
"""Lapicera""","""Librería""",1200,10,"""Normal"""
"""Mochila""","""Accesorios""",28500,2,"""Alto"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto"""
"""Regla""","""Librería""",900,6,"""Normal"""
"""Carpeta""","""Librería""",4200,5,"""Normal"""


La salida tiene `shape: (6, 5)` y agrega `nivel_precio` como columna de texto. `Mochila` y `Calculadora` reciben `"Alto"`; `Cuaderno`, `Lapicera`, `Regla` y `Carpeta` reciben `"Normal"`.

`then()` define el valor utilizado cuando la condición es verdadera y `otherwise()` cubre todos los demás casos.


## 4. La expresión puede guardarse

La construcción `when().then().otherwise()` también es una expresión de Polars, por lo que podemos almacenarla antes de utilizarla.


In [26]:
clasificacion_precio = (
    pl.when(pl.col("precio") > 10000)
    .then(pl.lit("Alto"))
    .otherwise(pl.lit("Normal"))
)

clasificacion_precio

<Expr ['.when([(col("precio")) > (dyn …'] at 0x7EC97374EF50>

La salida muestra un objeto `Expr`. `clasificacion_precio` todavía no contiene una columna calculada: almacena la expresión condicional completa para utilizarla después dentro de un contexto.


In [27]:
df.with_columns(
    nivel_precio=clasificacion_precio
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Normal"""
"""Lapicera""","""Librería""",1200,10,"""Normal"""
"""Mochila""","""Accesorios""",28500,2,"""Alto"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto"""
"""Regla""","""Librería""",900,6,"""Normal"""
"""Carpeta""","""Librería""",4200,5,"""Normal"""


Al evaluar `clasificacion_precio` dentro de `with_columns()`, obtenemos exactamente la misma columna `nivel_precio`: `"Alto"` para `Mochila` y `Calculadora`, y `"Normal"` para las cuatro filas restantes.


## 5. Una transformación numérica condicional

Las ramas no tienen que producir texto. Podemos calcular valores diferentes según la condición.

Aplicaremos un descuento del 10 % únicamente a los productos cuyo precio supera `10000`.


In [28]:
df.with_columns(
    precio_final=(
        pl.when(
            pl.col("precio") > 10000
        )
        .then(pl.col("precio") * 0.90)
        .otherwise(pl.col("precio"))
    )
)

producto,categoria,precio,cantidad,precio_final
str,str,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,3500.0
"""Lapicera""","""Librería""",1200,10,1200.0
"""Mochila""","""Accesorios""",28500,2,25650.0
"""Calculadora""","""Tecnología""",18500,3,16650.0
"""Regla""","""Librería""",900,6,900.0
"""Carpeta""","""Librería""",4200,5,4200.0


La nueva columna `precio_final` es de tipo `Float64`. Solo cambian las filas cuyo precio supera `10000`: `Mochila` pasa de `28500` a `25650.0` y `Calculadora` de `18500` a `16650.0`.

Las otras cuatro filas conservan el precio original, aunque representado como número de punto flotante para mantener un tipo común en toda la columna.


## 6. Conservar el valor original en los demás casos

Veamos otro ejemplo. Aplicamos una reducción del 5 % al precio cuando `cantidad` es mayor o igual que `6`.


In [29]:
df.with_columns(
    precio_ajustado=(
        pl.when(
            pl.col("cantidad") >= 6
        )
        .then(pl.col("precio") * 0.95)
        .otherwise(pl.col("precio"))
    )
)

producto,categoria,precio,cantidad,precio_ajustado
str,str,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,3500.0
"""Lapicera""","""Librería""",1200,10,1140.0
"""Mochila""","""Accesorios""",28500,2,28500.0
"""Calculadora""","""Tecnología""",18500,3,18500.0
"""Regla""","""Librería""",900,6,855.0
"""Carpeta""","""Librería""",4200,5,4200.0


La salida muestra que la reducción del 5 % se aplica únicamente cuando `cantidad >= 6`.

`Lapicera` pasa de `1200` a `1140.0` y `Regla` de `900` a `855.0`. Los demás productos conservan su precio original. La columna resultante es `Float64`.


## 7. Más de dos categorías

Ahora clasificamos los precios en tres niveles:

```text
precio > 15000  → "Alto"
precio > 3000   → "Medio"
resto           → "Bajo"
```


In [30]:
df.with_columns(
    nivel_precio=(
        pl.when(
            pl.col("precio") > 15000
        )
        .then(pl.lit("Alto"))
        .when(
            pl.col("precio") > 3000
        )
        .then(pl.lit("Medio"))
        .otherwise(pl.lit("Bajo"))
    )
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Medio"""
"""Lapicera""","""Librería""",1200,10,"""Bajo"""
"""Mochila""","""Accesorios""",28500,2,"""Alto"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto"""
"""Regla""","""Librería""",900,6,"""Bajo"""
"""Carpeta""","""Librería""",4200,5,"""Medio"""


La clasificación correcta produce tres niveles:

- `Mochila` y `Calculadora` → `"Alto"`;
- `Cuaderno` y `Carpeta` → `"Medio"`;
- `Lapicera` y `Regla` → `"Bajo"`.

La cadena se evalúa de arriba hacia abajo y utiliza la primera condición que resulta verdadera.


## 8. El orden de los `when()` importa

Para observar por qué, invertimos deliberadamente las dos condiciones.


In [31]:
df.with_columns(
    nivel_precio=(
        pl.when(
            pl.col("precio") > 3000
        )
        .then(pl.lit("Medio"))
        .when(
            pl.col("precio") > 15000
        )
        .then(pl.lit("Alto"))
        .otherwise(pl.lit("Bajo"))
    )
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Medio"""
"""Lapicera""","""Librería""",1200,10,"""Bajo"""
"""Mochila""","""Accesorios""",28500,2,"""Medio"""
"""Calculadora""","""Tecnología""",18500,3,"""Medio"""
"""Regla""","""Librería""",900,6,"""Bajo"""
"""Carpeta""","""Librería""",4200,5,"""Medio"""


Al invertir deliberadamente las condiciones aparece el problema que queríamos observar: `Mochila` y `Calculadora`, que deberían ser `"Alto"`, quedan clasificadas como `"Medio"`.

Como sus precios también son mayores que `3000`, satisfacen la primera condición y la evaluación no llega a la segunda. Esto confirma que el orden de los `when()` cambia el resultado cuando las reglas se superponen.


## 9. Construir intervalos explícitos

También podemos escribir condiciones que no se superpongan.


In [32]:
df.with_columns(
    nivel_precio=(
        pl.when(
            pl.col("precio") > 15000
        )
        .then(pl.lit("Alto"))
        .when(
            (pl.col("precio") > 3000)
            & (pl.col("precio") <= 15000)
        )
        .then(pl.lit("Medio"))
        .otherwise(pl.lit("Bajo"))
    )
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Medio"""
"""Lapicera""","""Librería""",1200,10,"""Bajo"""
"""Mochila""","""Accesorios""",28500,2,"""Alto"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto"""
"""Regla""","""Librería""",900,6,"""Bajo"""
"""Carpeta""","""Librería""",4200,5,"""Medio"""


La salida vuelve a producir la clasificación correcta: dos productos `"Alto"`, dos `"Medio"` y dos `"Bajo"`.

En esta versión la categoría `"Medio"` se define explícitamente mediante el intervalo `3000 < precio <= 15000`, por lo que las condiciones no se superponen.


## 10. Utilizar una condición compuesta

Las condiciones construidas con `&`, `|` o `~` pueden utilizarse directamente dentro de `when()`.

Clasificaremos como `"Prioritario"` a los productos de Librería cuya cantidad sea mayor o igual que `5`.


In [33]:
df.with_columns(
    estado=(
        pl.when(
            (pl.col("categoria") == "Librería")
            & (pl.col("cantidad") >= 5)
        )
        .then(pl.lit("Prioritario"))
        .otherwise(pl.lit("Normal"))
    )
)

producto,categoria,precio,cantidad,estado
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Normal"""
"""Lapicera""","""Librería""",1200,10,"""Prioritario"""
"""Mochila""","""Accesorios""",28500,2,"""Normal"""
"""Calculadora""","""Tecnología""",18500,3,"""Normal"""
"""Regla""","""Librería""",900,6,"""Prioritario"""
"""Carpeta""","""Librería""",4200,5,"""Prioritario"""


La nueva columna `estado` muestra `"Prioritario"` para `Lapicera`, `Regla` y `Carpeta`. Son las tres filas que cumplen simultáneamente las dos condiciones: pertenecer a `Librería` y tener `cantidad >= 5`.

Las otras tres filas reciben `"Normal"`.


## 11. La misma condición en `filter()` y en `when()`

Construimos una sola vez la condición:


In [34]:
condicion = pl.col("precio") > 10000

df.filter(condicion)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


Dentro de `filter()`, la condición `precio > 10000` reduce el DataFrame a `shape: (2, 4)` y conserva únicamente `Mochila` y `Calculadora`.


In [35]:
df.with_columns(
    nivel_precio=(
        pl.when(condicion)
        .then(pl.lit("Alto"))
        .otherwise(pl.lit("Normal"))
    )
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""Normal"""
"""Lapicera""","""Librería""",1200,10,"""Normal"""
"""Mochila""","""Accesorios""",28500,2,"""Alto"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto"""
"""Regla""","""Librería""",900,6,"""Normal"""
"""Carpeta""","""Librería""",4200,5,"""Normal"""


Dentro de `when()`, la misma condición ya no elimina filas. La salida mantiene `shape: (6, 5)` y utiliza la condición para asignar `"Alto"` a `Mochila` y `Calculadora` y `"Normal"` al resto.

La expresión es la misma; cambia el contexto y, por lo tanto, su función.


## 12. Varios predicados dentro de `when()`

Cuando todas las condiciones deben cumplirse, podemos proporcionarlas como predicados separados.


In [36]:
df.with_columns(
    destacado=(
        pl.when(
            pl.col("categoria") == "Librería",
            pl.col("cantidad") >= 5,
        )
        .then(pl.lit("Sí"))
        .otherwise(pl.lit("No"))
    )
)

producto,categoria,precio,cantidad,destacado
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""No"""
"""Lapicera""","""Librería""",1200,10,"""Sí"""
"""Mochila""","""Accesorios""",28500,2,"""No"""
"""Calculadora""","""Tecnología""",18500,3,"""No"""
"""Regla""","""Librería""",900,6,"""Sí"""
"""Carpeta""","""Librería""",4200,5,"""Sí"""


Los dos predicados pasados a `when()` se combinan mediante AND. La columna `destacado` contiene `"Sí"` para `Lapicera`, `Regla` y `Carpeta`, y `"No"` para las demás filas.

El resultado coincide con la condición compuesta que utilizamos anteriormente.


## 13. Qué ocurre si omitimos `otherwise()`

`otherwise()` es opcional. Observemos qué sucede cuando no lo escribimos.


In [37]:
df.with_columns(
    nivel_precio=(
        pl.when(
            pl.col("precio") > 10000
        )
        .then(pl.lit("Alto"))
    )
)

producto,categoria,precio,cantidad,nivel_precio
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,null
"""Lapicera""","""Librería""",1200,10,null
"""Mochila""","""Accesorios""",28500,2,"""Alto"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto"""
"""Regla""","""Librería""",900,6,null
"""Carpeta""","""Librería""",4200,5,null


La salida permite ver claramente qué ocurre al omitir `otherwise()`: `Mochila` y `Calculadora` reciben `"Alto"`, mientras que las otras cuatro filas contienen `null`.

Polars no completa automáticamente esos casos con otro valor. Si queremos que todas las filas reciban una categoría explícita, debemos proporcionar `otherwise()`.


## 14. Crear varias columnas condicionales

Podemos incluir más de una expresión condicional dentro de una misma llamada a `with_columns()`.


In [38]:
df.with_columns(
    nivel_precio=(
        pl.when(pl.col("precio") > 10000)
        .then(pl.lit("Alto"))
        .otherwise(pl.lit("Normal"))
    ),
    nivel_stock=(
        pl.when(pl.col("cantidad") >= 5)
        .then(pl.lit("Suficiente"))
        .otherwise(pl.lit("Bajo"))
    ),
)

producto,categoria,precio,cantidad,nivel_precio,nivel_stock
str,str,i64,i64,str,str
"""Cuaderno""","""Librería""",3500,4,"""Normal""","""Bajo"""
"""Lapicera""","""Librería""",1200,10,"""Normal""","""Suficiente"""
"""Mochila""","""Accesorios""",28500,2,"""Alto""","""Bajo"""
"""Calculadora""","""Tecnología""",18500,3,"""Alto""","""Bajo"""
"""Regla""","""Librería""",900,6,"""Normal""","""Suficiente"""
"""Carpeta""","""Librería""",4200,5,"""Normal""","""Suficiente"""


La salida tiene `shape: (6, 6)` y agrega dos clasificaciones independientes.

`nivel_precio` marca como `"Alto"` a `Mochila` y `Calculadora`. `nivel_stock` marca como `"Suficiente"` a `Lapicera`, `Regla` y `Carpeta`, cuyas cantidades son al menos `5`.

Como ambas expresiones dependen solo de columnas originales, pueden evaluarse dentro del mismo `with_columns()`.


## 15. Las ramas pueden contener expresiones

`then()` y `otherwise()` no están limitados a valores literales.

Aquí elegimos entre dos cálculos numéricos:


In [39]:
df.with_columns(
    valor_referencia=(
        pl.when(
            pl.col("cantidad") >= 5
        )
        .then(pl.col("precio") * pl.col("cantidad"))
        .otherwise(pl.col("precio"))
    )
)

producto,categoria,precio,cantidad,valor_referencia
str,str,i64,i64,i64
"""Cuaderno""","""Librería""",3500,4,3500
"""Lapicera""","""Librería""",1200,10,12000
"""Mochila""","""Accesorios""",28500,2,28500
"""Calculadora""","""Tecnología""",18500,3,18500
"""Regla""","""Librería""",900,6,5400
"""Carpeta""","""Librería""",4200,5,21000


La columna `valor_referencia` conserva tipo `Int64`. Cuando `cantidad >= 5`, utiliza `precio × cantidad`: `Lapicera` produce `12000`, `Regla` `5400` y `Carpeta` `21000`.

En las demás filas toma directamente el valor de `precio`: `3500`, `28500` y `18500`.

Esto muestra que `then()` y `otherwise()` pueden contener expresiones completas, no solo valores literales.


## 16. Una precaución importante con las ramas

Aunque la lógica se parezca a un `if` de Python, no debemos pensar que una rama inválida queda protegida simplemente porque su condición no se cumpla para una fila.

Las expresiones de las distintas ramas deben ser válidas por sí mismas.

Esto será especialmente importante más adelante, cuando trabajemos con conversiones de tipos, texto o valores faltantes. En este cuaderno todas nuestras ramas contienen expresiones válidas, por lo que no necesitamos resolver todavía casos problemáticos.


## 17. Integrar la transformación en una cadena

Terminamos combinando varios contextos.

Conservamos los productos cuyo precio supera `1000`, creamos una clasificación y finalmente mostramos solo tres columnas.


In [40]:
df.filter(
    pl.col("precio") > 1000
).with_columns(
    nivel_precio=(
        pl.when(
            pl.col("precio") > 10000
        )
        .then(pl.lit("Alto"))
        .otherwise(pl.lit("Normal"))
    )
).select(
    "producto",
    "precio",
    "nivel_precio",
)

producto,precio,nivel_precio
str,i64,str
"""Cuaderno""",3500,"""Normal"""
"""Lapicera""",1200,"""Normal"""
"""Mochila""",28500,"""Alto"""
"""Calculadora""",18500,"""Alto"""
"""Carpeta""",4200,"""Normal"""


La cadena final produce un DataFrame de `shape: (5, 3)`. `Regla` queda fuera porque su precio es `900` y no supera `1000`.

Entre las cinco filas restantes, `Mochila` y `Calculadora` reciben `"Alto"`, mientras que `Cuaderno`, `Lapicera` y `Carpeta` reciben `"Normal"`.

La secuencia completa queda visible en el resultado:

1. `filter()` elimina la fila que no supera `1000`;
2. `with_columns()` crea `nivel_precio`;
3. `when()` decide la categoría de cada fila restante;
4. `select()` deja únicamente `producto`, `precio` y `nivel_precio`.


## 18. Qué conviene recordar

`when()`, `then()` y `otherwise()` permiten construir expresiones cuyo resultado depende de una condición.

La estructura básica es:

```python
pl.when(
    condicion
).then(
    resultado_si_se_cumple
).otherwise(
    resultado_alternativo
)
```

En este cuaderno vimos que podemos:

- producir etiquetas de texto;
- realizar cálculos numéricos condicionales;
- conservar el valor original en `otherwise()`;
- encadenar varios `when()` para obtener más de dos categorías;
- utilizar condiciones compuestas;
- reutilizar una misma expresión en `filter()` y `when()`;
- pasar varios predicados cuando todos deben cumplirse;
- omitir `otherwise()` y observar la aparición de `null`;
- crear varias columnas condicionales en un mismo `with_columns()`.

También comprobamos que el orden de las condiciones puede cambiar el resultado cuando las reglas se superponen.


## Próximo paso

Ya podemos seleccionar columnas, filtrar filas, combinar condiciones, crear variables y construir transformaciones condicionales.

En el próximo capítulo utilizaremos estas herramientas junto con el ordenamiento. Aprenderemos a organizar los registros de menor a mayor o de mayor a menor, trabajar con varios criterios de orden, identificar extremos y comenzar a construir rankings.
